# 02 - Analise Exploratoria de Dados (EDA)

Responsavel: Integrante 2

Objetivo: explorar distribuicoes, correlacoes e padroes. Cada grafico deve ter TITULO descritivo e uma frase explicando o OBJETIVO analitico (a EDA alimenta a Secao 1 do dashboard).

In [ ]:
import os, subprocess

REPO_PATH = '/content/super-projeto-de-probabilidade-estatistica'

if os.path.exists(REPO_PATH):
    os.chdir(REPO_PATH)
    subprocess.run(['git', 'pull', 'origin', 'master'], check=True)

print('Diretório atual:', os.getcwd())

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

df = pd.read_csv('data/dataset_tratado.csv')

print('Shape:', df.shape)
print()
print(df.dtypes)
print()
df.head()

## 1. Distribuição da variável alvo

**Objetivo analítico:** verificar o balanceamento das classes antes da modelagem — desbalanceamento severo exigiria técnicas de resampling.

In [ ]:
CORES = {0: '#e74c3c', 1: '#2ecc71'}
LABELS = {0: 'Não Concluída', 1: 'Concluída'}
contagem = df['alvo'].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Barras
bars = axes[0].bar(
    [LABELS[k] for k in contagem.index],
    contagem.values,
    color=[CORES[k] for k in contagem.index],
    width=0.5, edgecolor='white'
)
axes[0].set_title('Contagem por Desfecho', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Quantidade de corridas')
for bar, v in zip(bars, contagem.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, v + 400,
                 f'{v:,}\n({v/len(df):.1%})', ha='center', va='bottom', fontsize=10)

# Pizza
axes[1].pie(
    contagem.values,
    labels=[LABELS[k] for k in contagem.index],
    autopct='%1.1f%%',
    colors=[CORES[k] for k in contagem.index],
    startangle=90, textprops={'fontsize': 11},
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[1].set_title('Proporção Concluída vs Não Concluída', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print(f'Desbalanceamento: {contagem[1]/contagem[0]:.2f}:1 (Concluída:Não Concluída)')
print('Conclusão: desbalanceamento moderado — não exige resampling para os modelos do projeto.')

## 2. Variáveis quantitativas

**Objetivo analítico:** entender a distribuição do tempo médio de chegada do veículo (`Avg VTAT`) e verificar se ele discrimina corridas concluídas de não concluídas. Booking Value, Ride Distance e CTAT foram removidas como leakage no notebook 01.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Histograma por alvo
for alvo_val, cor, label in [(1, '#2ecc71', 'Concluída'), (0, '#e74c3c', 'Não Concluída')]:
    subset = df[df['alvo'] == alvo_val]['Avg VTAT']
    axes[0].hist(subset, bins=30, alpha=0.6, color=cor, label=label, density=True)
axes[0].set_title('Distribuição de Avg VTAT por Desfecho', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Tempo médio de chegada (min)')
axes[0].set_ylabel('Densidade')
axes[0].legend()

# Boxplot por alvo (API nova do seaborn: hue + legend=False)
cores_str = {'0': '#e74c3c', '1': '#2ecc71'}
df_str = df.copy()
df_str['alvo'] = df_str['alvo'].astype(str)
sns.boxplot(data=df_str, x='alvo', y='Avg VTAT',
            hue='alvo', palette=cores_str, legend=False, ax=axes[1])
axes[1].set_xticklabels(['Não Concluída', 'Concluída'])
axes[1].set_title('Avg VTAT por Desfecho (Boxplot)', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Desfecho')
axes[1].set_ylabel('Tempo médio de chegada (min)')

# Distribuição de hora do dia
df['hora'].hist(bins=24, ax=axes[2], color='steelblue', edgecolor='white')
axes[2].set_title('Distribuição das Corridas por Hora do Dia', fontsize=11, fontweight='bold')
axes[2].set_xlabel('Hora')
axes[2].set_ylabel('Quantidade')

plt.tight_layout()
plt.show()

print('Avg VTAT — estatísticas por desfecho:')
print(df.groupby('alvo')['Avg VTAT'].describe().round(2).rename(index=LABELS))

## 3. Variáveis qualitativas

**Objetivo analítico:** conhecer a composição do dataset por tipo de veículo, período do dia, dia da semana e mês — essas variáveis serão usadas como features nos três modelos.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Tipo de veículo
ordem_veiculo = df['Vehicle Type'].value_counts().index
sns.countplot(data=df, y='Vehicle Type', order=ordem_veiculo,
              hue='Vehicle Type', palette='Blues_r', legend=False, ax=axes[0, 0])
axes[0, 0].set_title('Corridas por Tipo de Veículo', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Quantidade')
axes[0, 0].set_ylabel('')
for container in axes[0, 0].containers:
    axes[0, 0].bar_label(container, fmt='%,.0f', padding=3)

# Período do dia
ordem_periodo = ['manha', 'tarde', 'noite', 'madrugada']
cores_periodo = ['#f39c12', '#3498db', '#8e44ad', '#2c3e50']
contagem_periodo = df['periodo_dia'].value_counts().reindex(ordem_periodo, fill_value=0)
axes[0, 1].bar(contagem_periodo.index, contagem_periodo.values,
               color=cores_periodo, edgecolor='white')
axes[0, 1].set_title('Corridas por Período do Dia', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Período')
axes[0, 1].set_ylabel('Quantidade')
for i, v in enumerate(contagem_periodo.values):
    axes[0, 1].text(i, v + 200, f'{v:,}\n({v/len(df):.1%})', ha='center', fontsize=9)

# Dia da semana
nomes_dia = ['Seg', 'Ter', 'Qua', 'Qui', 'Sex', 'Sáb', 'Dom']
contagem_dia = df['dia_semana'].value_counts().sort_index()
axes[1, 0].bar(nomes_dia, contagem_dia.values, color='steelblue', edgecolor='white')
axes[1, 0].set_title('Corridas por Dia da Semana', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Dia')
axes[1, 0].set_ylabel('Quantidade')

# Mês
nomes_mes = ['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']
contagem_mes = df['mes'].value_counts().sort_index()
axes[1, 1].bar([nomes_mes[m-1] for m in contagem_mes.index],
               contagem_mes.values, color='teal', edgecolor='white')
axes[1, 1].set_title('Corridas por Mês', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Mês')
axes[1, 1].set_ylabel('Quantidade')

plt.tight_layout()
plt.show()

## 4. Relações com a variável alvo

**Objetivo analítico:** identificar quais features têm maior poder preditivo — taxas de conclusão muito diferentes entre categorias indicam features relevantes para os modelos.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Taxa de conclusão por tipo de veículo
taxa_veiculo = df.groupby('Vehicle Type')['alvo'].mean().sort_values(ascending=False)
bars = axes[0, 0].barh(taxa_veiculo.index, taxa_veiculo.values,
                        color=['#2ecc71' if v > 0.62 else '#e74c3c' for v in taxa_veiculo.values])
axes[0, 0].axvline(df['alvo'].mean(), color='k', linestyle='--', linewidth=1.2, label='Média geral')
axes[0, 0].set_title('Taxa de Conclusão por Tipo de Veículo', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Taxa de corridas concluídas')
axes[0, 0].legend()
axes[0, 0].set_xlim(0, 1)
for bar, v in zip(bars, taxa_veiculo.values):
    axes[0, 0].text(v + 0.01, bar.get_y() + bar.get_height()/2,
                    f'{v:.1%}', va='center', fontsize=9)

# Taxa de conclusão por período do dia
taxa_periodo = df.groupby('periodo_dia')['alvo'].mean().reindex(ordem_periodo)
axes[0, 1].bar(taxa_periodo.index, taxa_periodo.values, color=cores_periodo, edgecolor='white')
axes[0, 1].axhline(df['alvo'].mean(), color='k', linestyle='--', linewidth=1.2, label='Média geral')
axes[0, 1].set_title('Taxa de Conclusão por Período do Dia', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Período')
axes[0, 1].set_ylabel('Taxa de conclusão')
axes[0, 1].set_ylim(0, 1)
axes[0, 1].legend()
for i, v in enumerate(taxa_periodo.values):
    axes[0, 1].text(i, v + 0.015, f'{v:.1%}', ha='center', fontsize=10)

# Taxa de conclusão por hora do dia
taxa_hora = df.groupby('hora')['alvo'].mean()
axes[1, 0].plot(taxa_hora.index, taxa_hora.values, marker='o', color='steelblue', linewidth=2)
axes[1, 0].axhline(df['alvo'].mean(), color='k', linestyle='--', linewidth=1.2, label='Média geral')
axes[1, 0].fill_between(taxa_hora.index, taxa_hora.values, df['alvo'].mean(),
                         where=taxa_hora.values > df['alvo'].mean(),
                         alpha=0.2, color='green', label='Acima da média')
axes[1, 0].fill_between(taxa_hora.index, taxa_hora.values, df['alvo'].mean(),
                         where=taxa_hora.values <= df['alvo'].mean(),
                         alpha=0.2, color='red', label='Abaixo da média')
axes[1, 0].set_title('Taxa de Conclusão por Hora do Dia', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Hora')
axes[1, 0].set_ylabel('Taxa de conclusão')
axes[1, 0].set_xticks(range(0, 24))
axes[1, 0].legend(fontsize=8)

# Taxa de conclusão por dia da semana
taxa_dia = df.groupby('dia_semana')['alvo'].mean()
axes[1, 1].bar(nomes_dia, taxa_dia.values, color='mediumpurple', edgecolor='white')
axes[1, 1].axhline(df['alvo'].mean(), color='k', linestyle='--', linewidth=1.2, label='Média geral')
axes[1, 1].set_title('Taxa de Conclusão por Dia da Semana', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Dia')
axes[1, 1].set_ylabel('Taxa de conclusão')
axes[1, 1].set_ylim(0, 1)
axes[1, 1].legend()
for i, v in enumerate(taxa_dia.values):
    axes[1, 1].text(i, v + 0.01, f'{v:.1%}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

## 5. Correlações entre variáveis numéricas

**Objetivo analítico:** detectar multicolinearidade — features altamente correlacionadas entre si podem ser redundantes para os modelos de classificação.

In [ ]:
numericas = ['Avg VTAT', 'sem_vtat', 'hora', 'dia_semana', 'mes', 'alvo']
corr = df[numericas].corr()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Heatmap de correlação
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, vmin=-1, vmax=1, ax=axes[0],
            linewidths=0.5, annot_kws={'size': 10})
axes[0].set_title('Matriz de Correlação (Pearson)', fontsize=12, fontweight='bold')

# Correlação com o alvo (barras)
corr_alvo = corr['alvo'].drop('alvo').sort_values()
colors_corr = ['#e74c3c' if v < 0 else '#2ecc71' for v in corr_alvo.values]
axes[1].barh(corr_alvo.index, corr_alvo.values, color=colors_corr, edgecolor='white')
axes[1].axvline(0, color='k', linewidth=0.8)
axes[1].set_title('Correlação de Cada Feature com o Alvo', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Correlação de Pearson')
for i, v in enumerate(corr_alvo.values):
    axes[1].text(v + (0.002 if v >= 0 else -0.002), i,
                 f'{v:.3f}', va='center', ha='left' if v >= 0 else 'right', fontsize=9)

plt.tight_layout()
plt.show()

print('\nFeatures com maior correlação absoluta com o alvo:')
print(corr_alvo.abs().sort_values(ascending=False))

## 6. Insights principais

Resumo dos achados que orientam as decisões de modelagem nos notebooks 03 e 04.

In [ ]:
print('=' * 55)
print('INSIGHTS DA EDA')
print('=' * 55)

media_geral = df['alvo'].mean()
print(f'\n1. BALANCEAMENTO')
print(f'   {media_geral:.1%} das corridas são concluídas — desbalanceamento')
print(f'   moderado, aceitável sem resampling.')

print(f'\n2. TIPO DE VEÍCULO')
tv = df.groupby('Vehicle Type')['alvo'].mean().sort_values()
print(f'   Menor taxa: {tv.index[0]} ({tv.iloc[0]:.1%})')
print(f'   Maior taxa: {tv.index[-1]} ({tv.iloc[-1]:.1%})')
print(f'   → Feature relevante para Bayes e classificação.')

print(f'\n3. PERÍODO DO DIA')
tp = df.groupby('periodo_dia')['alvo'].mean()
print(f'   Variação entre períodos: {tp.min():.1%} a {tp.max():.1%}')
print(f'   → Padrão temporal existe; útil para Bayes manual.')

print(f'\n4. Avg VTAT')
vtat_c = df[df['alvo']==1]['Avg VTAT'].mean()
vtat_nc = df[df['alvo']==0]['Avg VTAT'].mean()
print(f'   Concluídas: {vtat_c:.1f} min | Não concluídas: {vtat_nc:.1f} min')
print(f'   → Diferença pequena pois nulos foram preenchidos com mediana.')

print(f'\n5. FLAG sem_vtat')
taxa_sem = df[df['sem_vtat']==1]['alvo'].mean()
taxa_com = df[df['sem_vtat']==0]['alvo'].mean()
print(f'   Taxa concluída COM motorista:  {taxa_com:.1%}')
print(f'   Taxa concluída SEM motorista:  {taxa_sem:.1%}')
print(f'   → sem_vtat é o preditor mais forte do dataset.')

print(f'\n6. CORRELAÇÕES')
print(f'   Sem multicolinearidade severa entre features.')
print(f'   hora, dia_semana e mes têm baixa correlação linear')
print(f'   com o alvo — mas relação não-linear pode existir.')
print('=' * 55)